In [ ]:
!pip install -U sentence-transformers peft accelerate transformers

# 1. Entraînement

In [ ]:
import math
import torch
import gc
from datasets import load_dataset
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, models
from peft import LoraConfig, get_peft_model
from sentence_transformers.losses.TripletLoss import TripletDistanceMetric

# --- NETTOYAGE MÉMOIRE AVANT DE COMMENCER ---
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

clear_gpu()

# --- CONFIGURATION ---
BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B" 
OUT_DIR = "Fe2x/qwen-lora-embedding"

# 1. Chargement des données
data_files = {"train": "train.jsonl", "validation": "valid.jsonl"}
ds = load_dataset("json", data_files=data_files)

def to_input_examples(split):
    return [
        InputExample(texts=[
            f"query: {r['query']}", 
            f"passage: {r['chunk_pos']}", 
            f"passage: {r['chunk_neg']}"
        ]) for r in split
    ]

train_examples = to_input_examples(ds["train"])
# Batch size de 2 est nécessaire pour éviter l'OOM avec le Triplet Loss sur 15Go
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2) 

# 2. Chargement du modèle
# On limite la longueur à 512 pour économiser la VRAM
word_embedding_model = models.Transformer(
    BASE_MODEL, 
    model_args={"torch_dtype": torch.float16, "trust_remote_code": True},
    max_seq_length=512 
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(), 
    pooling_mode="lasttoken" 
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

# 3. Configuration LoRA
lora_config = LoraConfig(
    r=16,                
    lora_alpha=32,       
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type=None       
)

# Appliquer LoRA
model[0].auto_model = get_peft_model(model[0].auto_model, lora_config)

# --- ACTIVATION GRADIENT CHECKPOINTING (CRITIQUE POUR LA VRAM) ---
model[0].auto_model.gradient_checkpointing_enable()
model[0].auto_model.print_trainable_parameters() 

# 4. Fonction de perte
loss = losses.TripletLoss(
    model=model,
    distance_metric=TripletDistanceMetric.COSINE, 
    triplet_margin=0.2
)

# 5. Entraînement
epochs = 1
warmup_steps = math.ceil(len(train_dataloader) * epochs * 0.1)

print("🚀 Lancement du fine-tuning (Version Low-VRAM)...")
try:
    model.fit(
        train_objectives=[(train_dataloader, loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        output_path=OUT_DIR,
        optimizer_params={'lr': 1e-4}, 
        show_progress_bar=True
    )
    print(f"Modèle LoRA sauvegardé avec succès dans : {OUT_DIR}")
except torch.cuda.OutOfMemoryError:
    print("Erreur OOM : Essayez de réduire batch_size à 1 ou de redémarrer le noyau Python.")

# 6. Sauvegarde
model.save(OUT_DIR)

## 2. Test & Validation

In [ ]:
from sentence_transformers import SentenceTransformer

base = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

import pandas as pd

valid = pd.read_json("valid.jsonl", lines=True)
valid.head()

In [ ]:
import numpy as np

def triplet_accuracy(model, df):
    ok = 0
    for _, r in df.iterrows():
        q = model.encode(r["query"], normalize_embeddings=True)
        p = model.encode(r["chunk_pos"], normalize_embeddings=True)
        n = model.encode(r["chunk_neg"], normalize_embeddings=True)

        if float(np.dot(q, p)) > float(np.dot(q, n)):
            ok += 1
    return ok / len(df)

acc = triplet_accuracy(model, valid)
print(f"Triplet accuracy (valid): {acc:.3f}")
print("Base acc:", triplet_accuracy(base, valid))
print("FT   acc:", triplet_accuracy(model, valid))

In [ ]:

print("Base acc:", triplet_accuracy(base, valid))
print("FT   acc:", triplet_accuracy(model, valid))

In [ ]:
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


valid = load_jsonl("valid.jsonl")

from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformer
import numpy as np

def build_full_corpus_ir(triplets):
    queries = {}
    relevant_docs = {}
    corpus = {}

    # corpus = tous les chunks positifs uniques
    for t in triplets:
        pid = str(t["pos_id"])
        corpus[pid] = t["chunk_pos"]

    # queries + relevances
    for i, t in enumerate(triplets):
        qid = str(i)
        queries[qid] = t["query"]
        relevant_docs[qid] = {str(t["pos_id"])}

    return queries, corpus, relevant_docs


queries, corpus, relevant_docs = build_full_corpus_ir(valid)

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    show_progress_bar=True,
    name="valid_ir",
)

print("Base IR:")
base_scores = evaluator(base)

print("FT IR:")
ft_scores = evaluator(model)

print(base_scores)
print(ft_scores)




In [ ]:
import torch.nn.functional as F

def check_robustness(model, query, pos, neg):
    # Encodage
    q_emb = model.encode(f"query: {query}", convert_to_tensor=True)
    p_emb = model.encode(f"passage: {pos}", convert_to_tensor=True)
    n_emb = model.encode(f"passage: {neg}", convert_to_tensor=True)
    
    # Calcul des scores de cosinus
    score_pos = F.cosine_similarity(q_emb.unsqueeze(0), p_emb.unsqueeze(0)).item()
    score_neg = F.cosine_similarity(q_emb.unsqueeze(0), n_emb.unsqueeze(0)).item()
    
    print(f"Query: {query[:50]}...")
    print(f"  Pos Score: {score_pos:.4f}")
    print(f"  Neg Score: {score_neg:.4f}")
    print(f"  Différence (Marge): {score_pos - score_neg:.4f}\n")

# Test sur les 3 premiers exemples de la validation
for i in range(3):
    sample = ds["validation"][i]
    check_robustness(model, sample['query'], sample['chunk_pos'], sample['chunk_neg'])